In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00


In [ ]:
# run after restarting runtime
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print(f"CLIP loaded on {device} ✓")

100%|███████████████████████████████████████| 338M/338M [00:06<00:00, 51.5MiB/s]


CLIP loaded on cpu ✓


In [ ]:
# reload everything after restart runtime
import os, joblib, numpy as np, shap
import pandas as pd

BASE  = "/content/drive/MyDrive/capstone"

from google.colab import drive
drive.mount("/content/drive")

m8    = joblib.load(f"{BASE}/models/m8_longform_clip.pkl")
sc_m8 = joblib.load(f"{BASE}/models/m8_longform_scaler.pkl")
m5    = joblib.load(f"{BASE}/models/m5_shorts_clip.pkl")
sc_m5 = joblib.load(f"{BASE}/models/m5_shorts_scaler.pkl")

CLIP_FEATURES = [f"clip_{i}" for i in range(50)]
print("✅ Everything reloaded — ready to run demo")

Mounted at /content/drive
✅ Everything reloaded — ready to run demo


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image as PILImage
import io
import requests as req
import numpy as np
from sklearn.decomposition import PCA

# ── Anthropic API key
ANTHROPIC_API_KEY = "****"   # remove key for github
# ── format selector + upload widget
format_selector = widgets.ToggleButtons(
    options=["Long-form", "Short"],
    description="Video type:",
    button_style="info"
)
upload = widgets.FileUpload(accept="image/*", multiple=False,
                            description="Upload Thumbnail")
output = widgets.Output()


def get_ai_suggestions(score, format_label, views_per_day, top_dims):

    # ── try Claude API first
    try:
        model_knowledge = """
        You are a YouTube thumbnail analyst. Here is what a virality predictor model
        learned from 4,710 YouTube thumbnails:

        KEY FINDINGS:
        - CLIP semantic features predict virality far better than raw pixel features
        - Long-form thumbnails: R2=0.74 (highly predictable)
        - Shorts thumbnails: R2=0.52 (harder to predict)
        - Dark thumbnails consistently underperform — dark_ratio was the top negative predictor
        - High edge density (cluttered/busy thumbnails) hurts performance
        - Colorfulness, contrast, sharpness and saturation all help virality
        - Shorts and long-form respond to completely different visual patterns
        - Long-form top signals: clip_3, clip_1 (production quality, clear subject framing)
        - Shorts top signals: clip_11, clip_7, clip_0 (dynamic, energetic, motion cues)
        """

        prompt = f"""
        A creator uploaded a {format_label} thumbnail.
        MODEL RESULTS:
        - Virality score: {score:.2f} (log views/day)
        - Estimated views per day: {views_per_day:,.0f}
        - Top influential visual dimensions: {top_dims}

        Give the creator:
        1. One sentence summary of predicted performance
        2. Two specific things to IMPROVE (based on what hurts virality)
        3. Two specific things to KEEP or AMPLIFY (based on what helps virality)
        4. One tip specific to {format_label} thumbnails
        Be specific, practical and encouraging. Under 200 words.
        """

        response = req.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "Content-Type": "application/json",
                "x-api-key": ANTHROPIC_API_KEY,
                "anthropic-version": "2023-06-01"
            },
            json={
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 400,
                "system": model_knowledge,
                "messages": [{"role": "user", "content": prompt}]
            },
            timeout=10
        )

        data = response.json()

        # if API returns any error — credits, auth, rate limit — fall through
        if "error" in data or "content" not in data:
            raise ValueError(data.get("error", {}).get("message", "API unavailable"))

        print("🤖 AI Suggestions (powered by Claude):\n")
        return data["content"][0]["text"]

    # ── fallback — runs automatically if API fails for any reason
    except Exception as e:

        print("🤖 API credit is nlow manual suggestions (model-based analysis):\n")  # no mention of fallback

        if score >= 9:
            performance = "strong — this thumbnail is predicted to perform well above average"
        elif score >= 7:
            performance = "moderate — this thumbnail should perform around average"
        else:
            performance = "below average — there is room to improve this thumbnail"

        if format_label == "Long-form":
            improve = [
                "Reduce dark areas — dark_ratio was the single strongest negative predictor. Brighten the subject or add a contrasting background.",
                "Simplify the composition — high edge density consistently hurt long-form virality. Try a cleaner layout with one clear focal point."
            ]
            keep = [
                "Maintain sharpness and focus — crisp, well-lit subjects outperform blurry or low-contrast ones.",
                "Keep strong color contrast — saturation and colorfulness were both positive predictors. Vivid colors help thumbnails stand out in the feed."
            ]
            specific = "Top long-form thumbnails had a single well-lit subject against a clean background with bold, readable text."

        else:
            improve = [
                "Add energy and motion — the dominant CLIP dimensions for Shorts encode dynamic, action-oriented cues. Static or posed shots underperform.",
                "Avoid dark or muted tones — bright, saturated frames perform significantly better for Shorts."
            ]
            keep = [
                "Keep faces front and center if present — face presence is a consistent positive signal across both formats.",
                "Maintain high colorfulness — vivid thumbnails outperform muted ones even when displayed small on screen."
            ]
            specific = "Top Shorts used bold close-up shots with high contrast, often with expressive facial reactions and text overlays."

        return f"""📊 Performance prediction: {performance} (score: {score:.1f}, ~{views_per_day:,.0f} views/day estimated).

🔧 Two things to IMPROVE:
- {improve[0]}
- {improve[1]}

✅ Two things to KEEP:
- {keep[0]}
- {keep[1]}

💡 {format_label} tip:
{specific}"""


def predict_thumbnail(change):
    if not upload.value:
        return

    with output:
        clear_output()

        # ── get format from selector
        format_label = format_selector.value                # "Long-form" or "Short"
        print(f"📺 Format selected: {format_label}")

        uploaded_file = list(upload.value.values())[0]
        img_bytes     = uploaded_file["content"]
        img           = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")

        display(img.resize((320, 180)))
        print("🔍 Analysing thumbnail...")

        # ── CLIP encoding
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model.encode_image(img_tensor)
            features = features.cpu().numpy().flatten()

        # ── PCA reduce to 50 dims
        raw_embeddings = np.load(
            f"{BASE}/Project2data/embeddings/clip_embeddings_512.npy")
        pca_fit = PCA(n_components=50, random_state=42)
        pca_fit.fit(raw_embeddings)
        features_50 = pca_fit.transform(features.reshape(1, -1))

        # ── pick model based on selector
        if format_label == "Short":
            from sklearn.preprocessing import StandardScaler
            scaled   = sc_m5.transform(
                pd.DataFrame(features_50, columns=CLIP_FEATURES))
            score    = m5.predict(scaled)[0]
            top_dims = "clip_11, clip_7, clip_0, clip_9, clip_26"
            model_name = "M5 (Shorts)"
        else:
            scaled   = sc_m8.transform(
                pd.DataFrame(features_50, columns=CLIP_FEATURES))
            score    = m8.predict(scaled)[0]
            top_dims = "clip_3, clip_1, clip_4, clip_6, clip_2"
            model_name = "M8 (Long-form)"

        views_per_day = np.expm1(score)

        if score >= 9:
            tier = "🔥 High virality"
        elif score >= 7:
            tier = "⚡ Medium virality"
        else:
            tier = "📉 Low virality"


        print("\n" + "─" * 45)
        print(f"  Format:           {format_label}")
        print(f"  Model used:       {model_name}")
        print(f"  Virality score:   {score:.2f}")
        print(f"  Est. views/day:   {views_per_day:,.0f}")
        print(f"  Tier:             {tier}")
        print("─" * 45)


        print("\n🤖 AI Suggestions based on model training:\n")
        suggestions = get_ai_suggestions(score, format_label,
                                         views_per_day, top_dims)
        print(suggestions)
        print("\n" + "─" * 45)



upload.observe(predict_thumbnail, names="value")

print("📸 YouTube Thumbnail Virality Predictor\n")
print("Step 1 — select your video type")
print("Step 2 — upload your thumbnail\n")
display(format_selector, upload, output)

📸 YouTube Thumbnail Virality Predictor

Step 1 — select your video type
Step 2 — upload your thumbnail



ToggleButtons(button_style='info', description='Video type:', options=('Long-form', 'Short'), value='Long-form…

FileUpload(value={}, accept='image/*', description='Upload Thumbnail')

Output()